# Title vs Tracks Comparison — PCA + Fuzzy Clustering

This is a corrected version of `title_to_tracks_comparison.ipynb`.
Both embedding spaces are reduced to 50 PCA dimensions before fuzzy c-means
so that Euclidean distances are meaningful and memberships are non-uniform.
Cluster labels are then aligned with the Hungarian algorithm before computing
KL divergence, and ARI/NMI are also reported for direct comparison with Susans results

# Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_BASE           = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data"
PROJECT_BASE         = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Track vs Title"
EMBEDDINGS_PKL       = f"{DRIVE_BASE}/embeddings.pkl"
CLUSTERS_OUTPUT_DIR  = f"{PROJECT_BASE}/pca_clusters"
REPO_DIR             = "/content/PlaylistClustering"

os.makedirs(CLUSTERS_OUTPUT_DIR, exist_ok=True)
print("Paths configured.")

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/PlaylistClustering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%pip install -r {REPO_DIR}/requirements.txt
%pip install umap-learn -q

In [ ]:
cluster_alts_path = os.path.join(REPO_DIR, "PlaylistRecsysUpgrade", "clustering", "cluster_alts.py")

with open(cluster_alts_path, 'r') as f:
    content = f.read()

# Replace the incorrect import statement
content = content.replace('from skfuzzy import cmeans, cmeans_predict', 'import skfuzzy as fuzz')

# Replace function calls
content = content.replace('cmeans(', 'fuzz.cmeans(')
content = content.replace('cmeans_predict(', 'fuzz.cmeans_predict(')

with open(cluster_alts_path, 'w') as f:
    f.write(content)

print(f"Successfully modified {cluster_alts_path} to correct skfuzzy imports.")

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "cluster_alts",
    os.path.join(REPO_DIR, "PlaylistRecsysUpgrade", "clustering", "cluster_alts.py"),
)
cluster_alts = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cluster_alts)

print("cluster_alts loaded.")

# Fix Pandas Version

In [ ]:
# # @title
# import subprocess, sys

# # Upgrade pandas to latest
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pandas", "-q"])

# # Restart runtime so the new pandas version is actually loaded
# import os
# os.kill(os.getpid(), 9)  # Forces a runtime restart — re-run all cells after this

# Imports

In [ ]:
import os
import pickle
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from scipy.stats import entropy
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from skfuzzy import cmeans

# Load Embeddings

In [ ]:
with open(EMBEDDINGS_PKL, 'rb') as f:
    data = pickle.load(f)

print("Full DataFrame shape:", data.shape)
print("Columns:", data.columns.tolist())
print(data.head())

In [ ]:
# Diagnostic: check whether embeddings are L2-normalized.
# SBERT and most sentence-transformer models output unit-norm vectors by default.
# If norms are all ~1.0, Euclidean distances concentrate around sqrt(2) regardless
# of dimensionality — explaining why PCA alone doesn't fix uniform memberships.
title_sample  = np.stack(data['title_embedding'].values[:200])
tracks_sample = np.stack(data['tracks_embedding'].values[:200])

title_norms  = np.linalg.norm(title_sample,  axis=1)
tracks_norms = np.linalg.norm(tracks_sample, axis=1)

print(f"Title  embedding norms — mean: {title_norms.mean():.4f}, std: {title_norms.std():.4f}")
print(f"Tracks embedding norms — mean: {tracks_norms.mean():.4f}, std: {tracks_norms.std():.4f}")
print()
if title_norms.std() < 0.01:
    print("⚠  Title embeddings appear L2-normalized (unit sphere).")
    print("   Euclidean distances will concentrate around √2 — this is why PCA alone")
    print("   does not fix uniform fuzzy memberships. UMAP is used below instead.")
else:
    print("Title embeddings are NOT unit-normalized — PCA alone may be sufficient.")

# Dimensionality Reduction — PCA then UMAP

If embeddings are L2-normalized (unit sphere), PCA alone cannot fix distance
concentration because all points remain geometrically equidistant regardless
of which linear subspace we project onto.

**Pipeline:**
1. **PCA to 50-d** — fast, removes low-variance noise, keeps computation tractable.
2. **UMAP to 15-d** — explicitly optimizes to preserve local neighborhood structure,
   "unrolling" the hypersphere into a Euclidean space where inter-point distances
   vary meaningfully. Fuzzy c-means then sees real geometric separation between
   clusters and produces non-uniform memberships.

**Memory strategy:** UMAP's k-NN graph is O(n × n_neighbors) and crashes Colab on
~1M points. We fit UMAP on a random subsample (`UMAP_FIT_SAMPLE`), then call
`.transform()` on the full dataset in batches. `.transform()` only needs to locate
each new point in the learned manifold — far cheaper than re-fitting.

`min_dist=0.0` is used for clustering (tightens intra-cluster packing).
`n_neighbors=15` is the standard default.

In [ ]:
import umap

PCA_COMPONENTS  = 50
UMAP_COMPONENTS = 15
UMAP_FIT_SAMPLE = 50_000   # rows used to *fit* UMAP; full dataset is transformed in batches
TRANSFORM_BATCH = 10_000   # rows per .transform() call to cap peak RAM

rng = np.random.default_rng(42)

print("Stacking embeddings...")
title_embeddings  = np.stack(data['title_embedding'].values)   # (n, 384)
tracks_embeddings = np.stack(data['tracks_embedding'].values)  # (n, 384)
n_total = len(title_embeddings)
print(f"  Title  raw shape: {title_embeddings.shape}")
print(f"  Tracks raw shape: {tracks_embeddings.shape}")

# ── Stage 1: PCA ────────────────────────────────────────────────────────────
print("\nRunning PCA...")
pca_title = PCA(n_components=PCA_COMPONENTS, random_state=42)
title_pca = pca_title.fit_transform(title_embeddings)
print(f"  Title  PCA: {pca_title.explained_variance_ratio_.sum():.1%} variance retained")
del title_embeddings   # free raw embeddings immediately

pca_tracks = PCA(n_components=PCA_COMPONENTS, random_state=42)
tracks_pca = pca_tracks.fit_transform(tracks_embeddings)
print(f"  Tracks PCA: {pca_tracks.explained_variance_ratio_.sum():.1%} variance retained")
del tracks_embeddings

# ── Stage 2: UMAP — fit on subsample, transform full dataset in batches ─────
fit_idx = rng.choice(n_total, size=min(UMAP_FIT_SAMPLE, n_total), replace=False)

def umap_fit_transform_batched(data_pca, fit_idx, tag):
    """Fit UMAP on `fit_idx` rows, then transform the full array in batches."""
    reducer = umap.UMAP(
        n_components=UMAP_COMPONENTS,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        low_memory=True,   # avoids materialising the full O(n²) distance matrix
        random_state=42,
        verbose=False,
    )
    print(f"  [{tag}] Fitting UMAP on {len(fit_idx):,} samples...")
    reducer.fit(data_pca[fit_idx])

    print(f"  [{tag}] Transforming {n_total:,} points in batches of {TRANSFORM_BATCH:,}...")
    parts = []
    for start in range(0, n_total, TRANSFORM_BATCH):
        batch = data_pca[start : start + TRANSFORM_BATCH]
        parts.append(reducer.transform(batch))
        print(f"    {min(start + TRANSFORM_BATCH, n_total):,} / {n_total:,}", end="\r")
    print()
    return np.vstack(parts)

print("\nRunning UMAP on title embeddings...")
title_umap = umap_fit_transform_batched(title_pca, fit_idx, "title")
del title_pca
print(f"  Title  UMAP output shape: {title_umap.shape}")

print("\nRunning UMAP on track embeddings...")
tracks_umap = umap_fit_transform_batched(tracks_pca, fit_idx, "tracks")
del tracks_pca
print(f"  Tracks UMAP output shape: {tracks_umap.shape}")

# Fuzzy C-Means on PCA-Reduced Title Embeddings

In [ ]:
NUM_CLUSTERS = 50

fkmeans_titles_output = os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_title_clusters_umap.pkl")

print("Fitting Fuzzy C-Means on UMAP-reduced title embeddings...")
# cmeans expects (features, n_samples), so transpose
cntr_t, u_t, _, _, _, _, fpc_t = cmeans(
    title_umap.T, NUM_CLUSTERS, 2, error=0.005, maxiter=1000
)
print(f"  FPC (fuzzy partition coefficient): {fpc_t:.4f}")
print(f"  FPC=1.0 → crisp  |  FPC=1/k={1/NUM_CLUSTERS:.4f} → fully uniform (broken)")

# u_t shape: (NUM_CLUSTERS, n_samples) → transpose to (n_samples, NUM_CLUSTERS)
title_cluster_probs = u_t.T

print("\nSample membership row:")
print(np.round(title_cluster_probs[0], 4))
print(f"  Max membership: {title_cluster_probs[0].max():.4f}  (was ~0.02 before UMAP)")

title_clusters_df = pd.DataFrame({
    'playlist_title': data['playlist_title'],
    'title_cluster_probs': list(title_cluster_probs)
})

title_clusters_df.to_pickle(fkmeans_titles_output)
print(f"\nSaved to {fkmeans_titles_output}")
display(title_clusters_df.head())

# Fuzzy C-Means on PCA-Reduced Track Embeddings

In [ ]:
fkmeans_tracks_output = os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_tracks_clusters_umap.pkl")

print("Fitting Fuzzy C-Means on UMAP-reduced track embeddings...")
cntr_k, u_k, _, _, _, _, fpc_k = cmeans(
    tracks_umap.T, NUM_CLUSTERS, 2, error=0.005, maxiter=1000
)
print(f"  FPC (fuzzy partition coefficient): {fpc_k:.4f}")

tracks_cluster_probs = u_k.T

print("\nSample membership row:")
print(np.round(tracks_cluster_probs[0], 4))
print(f"  Max membership: {tracks_cluster_probs[0].max():.4f}")

tracks_clusters_df = pd.DataFrame({
    'playlist_title': data['playlist_title'],
    'tracks_cluster_probs': list(tracks_cluster_probs)
})

tracks_clusters_df.to_pickle(fkmeans_tracks_output)
print(f"\nSaved to {fkmeans_tracks_output}")
display(tracks_clusters_df.head())

# Align Cluster Labels (Hungarian Algorithm)

Two independent fuzzy c-means runs produce cluster indices that have no
correspondence to each other — title cluster 7 and track cluster 7 are
unrelated groups. Before computing per-sample KL divergence we must find
the permutation of the track clusters that best matches the title clusters.

We use the **Hungarian algorithm** on a cost matrix where
`cost[i, j] = -sum_n min(p_n_i, q_n_j)` — negative because
`linear_sum_assignment` minimises cost.

In [ ]:
p_matrix = np.vstack(title_clusters_df['title_cluster_probs'].values)   # (n, k)
q_matrix = np.vstack(tracks_clusters_df['tracks_cluster_probs'].values) # (n, k)

print("Shapes — p (title):", p_matrix.shape, "  q (tracks):", q_matrix.shape)

# Build cost matrix: overlap between title-cluster i and track-cluster j
# p.T @ q gives (k, k) where entry [i,j] = sum_n p_ni * q_nj
# Using dot-product overlap is faster than computing min() for each pair.
cost = -p_matrix.T @ q_matrix  # (k, k)

row_ind, col_ind = linear_sum_assignment(cost)
# col_ind[i] is the track-cluster index that best matches title-cluster i
q_aligned = q_matrix[:, col_ind]  # reorder track cluster columns

print(f"Label alignment complete. Example mapping (first 10): title cluster → track cluster")
for ti, ki in zip(row_ind[:10], col_ind[:10]):
    print(f"  {ti} → {ki}")

# KL Divergence on Aligned Distributions

In [ ]:
print("Computing KL Divergence...")

epsilon = 1e-10
p_safe = np.clip(p_matrix, epsilon, 1.0)
q_safe = np.clip(q_aligned, epsilon, 1.0)

p_safe /= p_safe.sum(axis=1, keepdims=True)
q_safe /= q_safe.sum(axis=1, keepdims=True)

kl_divergences = entropy(p_safe, q_safe, axis=1)

title_clusters_df['kl_divergence'] = kl_divergences
merged_df = title_clusters_df.copy()

print(f"KL Divergence — mean: {kl_divergences.mean():.4f}, "
      f"median: {np.median(kl_divergences):.4f}, "
      f"max: {kl_divergences.max():.4f}")
display(merged_df[['playlist_title', 'kl_divergence']].head())

# Hard-Assignment Metrics (ARI & NMI)

These operate on `argmax` labels and are permutation-invariant, so they
do **not** require the Hungarian alignment step above.

In [ ]:
title_hard  = np.argmax(p_matrix, axis=1)
tracks_hard = np.argmax(q_matrix, axis=1)

ari = adjusted_rand_score(title_hard, tracks_hard)
nmi = normalized_mutual_info_score(title_hard, tracks_hard)

print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Info (NMI): {nmi:.4f}")
print()
print("ARI=0 means the two clusterings agree no better than random chance.")
print("ARI=1 means perfect agreement.")
print("Your teammate's k-means ARI was 0.0118 — compare directly here.")

# Distribution of KL Divergence

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(merged_df['kl_divergence'], bins=50, kde=True, color='purple')
plt.title('Distribution of KL Divergence between Title and Track Clusters (PCA-corrected)')
plt.xlabel('KL Divergence')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

print("Top 5 Playlists with HIGHEST KL Divergence (Most Mismatched):")
display(merged_df.nlargest(5, 'kl_divergence')[['playlist_title', 'kl_divergence']])

print("\nTop 5 Playlists with LOWEST KL Divergence (Most Matched):")
display(merged_df.nsmallest(5, 'kl_divergence')[['playlist_title', 'kl_divergence']])

# Cluster-Level KL Divergence Analysis

In [ ]:
print("Assigning playlists to dominant title clusters...")
merged_df['dominant_cluster'] = title_hard

cluster_kl_stats = merged_df.groupby('dominant_cluster')['kl_divergence'].agg(
    mean_kl='mean',
    median_kl='median',
    playlist_count='count'
).reset_index().sort_values(by='median_kl')

print("\nTop 10 Clusters with LOWEST Median KL Divergence (Most Matched):")
display(cluster_kl_stats.head(10))

print("\nTop 10 Clusters with HIGHEST Median KL Divergence (Most Mismatched):")
display(cluster_kl_stats.tail(10))

plt.figure(figsize=(14, 6))
sns.barplot(
    x='dominant_cluster',
    y='median_kl',
    data=cluster_kl_stats,
    order=cluster_kl_stats['dominant_cluster'],
    palette='viridis'
)
plt.title('Median KL Divergence by Dominant Title Cluster (PCA-corrected)')
plt.xlabel('Cluster ID (Sorted by Median KL Divergence)')
plt.ylabel('Median KL Divergence')
plt.xticks(
    ticks=range(len(cluster_kl_stats)),
    labels=cluster_kl_stats['dominant_cluster'],
    rotation=90
)
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()